# Tool Use and MCP: a hands-on course

*Lecture notebook. Read the markdown like lecture notes, run the code like a lab. Everything runs offline.*

## Why this matters
A language model on its own can only produce text. **Tools** let it act: search, calculate, read files, call APIs.
**MCP (Model Context Protocol)** is an open standard so that any AI application can use any tool server without custom glue code for each pair.

## Learning objectives
1. Explain the **tool-call loop** and what the model, the harness and the tool each do.
2. **Validate** tool arguments and return errors the model can learn from.
3. Apply **tool design** principles (descriptions, outputs, errors, parallel calls).
4. Explain MCP's **architecture**: host, client, server, and the three primitives (tools, resources, prompts).
5. Speak MCP **by hand** (JSON-RPC over stdio), then use the **official SDK**.
6. Recognize **security** risks: prompt injection through tool results, and defend with approval gates and least privilege.

## The big picture
```
 +----------------- HOST (your AI app) -----------------+
 |                                                      |
 |   LLM  <---->  harness  <---->  MCP client(s)        |
 |                                                      |
 +---------------------------+--------------------------+
                             |
     JSON-RPC 2.0 over stdio (local) or HTTP (remote)
                             |
             +---------------+---------------+
             |                               |
             v                               v
   +-------------------+           +-------------------+
   |    MCP SERVER A   |           |    MCP SERVER B   |
   | tools             |           | tools             |
   | resources         |           | resources         |
   | prompts           |           | prompts           |
   +-------------------+           +-------------------+
    (notes, files, GitHub...)       (database, calendar...)
```
Without a standard, N apps times M tools needs N x M integrations. With MCP it is N + M.

## How to use this notebook
- Companion code: `tooluse.py` (loop, validator, MCP client), `mini_mcp_server.py` (an MCP server from scratch),
  `sdk_server.py` (the same idea with the official SDK).
- The "model" is **scripted** so results are deterministic.
- **Slide deck** for this course: https://claude.ai/artifact/1Y2oPW5Z6MrLcATas7o41f

In [ ]:
# If this cell errors, the notebook is probably not using the project's kernel.
# Pick "Python (tool-mcp)" (top right in VS Code / Jupyter), then run it again.
import json, sys, asyncio
try:
    import pandas as pd
    from tooluse import *
except ImportError as e:
    raise SystemExit(f"Missing package or file: {e}. Select the 'Python (tool-mcp)' kernel and run from the Tool_Use_MCP_Learning folder.")

## 1. Anatomy of a tool call

**Concept.** The model never runs code. It emits a **structured request** ("call `add` with `a=2, b=3`"), the **harness** runs it, and the
**result** goes back to the model as a new message. Repeat until the model answers in plain text.

```
user      : "What is 2 + 3?"
assistant : tool_call  add {a: 2, b: 3}          <- the model REQUESTS
tool      : "5"                                   <- the harness RUNS and reports
assistant : "2 + 3 = 5"                           <- final answer
```

A tool definition has three parts the model sees: a **name**, a **description**, and a **JSON Schema** for the inputs.
(Provider APIs differ in field names, but this is the common shape.)

In [ ]:
registry = ToolRegistry()
registry.register("add", "Add two numbers and return the sum.",
                  {"type": "object", "properties": {"a": {"type": "number"}, "b": {"type": "number"}}, "required": ["a", "b"]},
                  lambda a, b: a + b)
registry.register("word_count", "Count the words in a piece of text.",
                  {"type": "object", "properties": {"text": {"type": "string"}}, "required": ["text"]},
                  lambda text: len(text.split()))

print(json.dumps(registry.definitions()[0], indent=2))   # exactly what the model is shown for one tool

### Worked example: how the registry is used, in three steps

**Step 1. What the model is shown.** `registry.definitions()` returns each tool's name, description and schema. The Python function stays hidden. In a real API this list is sent with **every** request.

In [ ]:
print(json.dumps(registry.definitions(), indent=1))

**Step 2. What the harness does when the model asks for a tool.** `registry.call(name, args)` validates the arguments, runs the function, and returns a pair: `(text, is_error)`.
Look at the last two calls: **errors do not crash the program**. They come back as readable text the model can learn from.

In [ ]:
requests = [
    ("add",        {"a": 2, "b": 3}),                          # valid
    ("word_count", {"text": "tool use is a loop"}),            # valid
    ("add",        {"a": "two", "b": 3}),                      # wrong type
    ("multiply",   {"a": 2, "b": 3}),                          # a tool that does not exist
]
for name, args in requests:
    text, is_error = registry.call(name, args)
    print(f"call {name}({args})\n   -> {text!r}   is_error={is_error}\n")

**Step 3. How it fits into a conversation.** `run_tool_loop` ties the registry to a model. The model below is a tiny scripted stand-in for an LLM: it asks for `add(2, 3)`, then writes a final answer **using the tool result**.

```
user      : What is 2 + 3?
assistant : tool_call add {a: 2, b: 3}      <- the model REQUESTS
tool      : "5"                             <- registry.call() RAN it
assistant : "2 + 3 = 5"                     <- final answer, written from the result
```

In [ ]:
class TwoStepModel:
    """Stands in for the LLM. Turn 1: request a tool. Turn 2: answer using the result."""
    def next_turn(self, messages, tools):
        results = [m for m in messages if m["role"] == "tool"]
        if not results:
            return {"calls": [{"id": "c1", "name": "add", "args": {"a": 2, "b": 3}}]}   # ask for a tool
        return {"final": f"2 + 3 = {results[-1]['content']}"}                            # answer from the tool result

result = run_tool_loop(TwoStepModel(), "What is 2 + 3?", registry, verbose=False)
for m in result["messages"]:
    print({k: v for k, v in m.items() if k in ("role", "content", "tool_calls", "name")})
print("final answer:", result["answer"])

**Takeaway.** The model never runs code. It **names a tool and arguments**, the harness runs `registry.call`, and the result returns as a `tool` message.

## 2. Validation and errors: make failure recoverable

**Concept.** Models make mistakes: wrong types, missing fields, invented tool names. The harness should:
1. **Validate** arguments against the schema *before* running the tool.
2. Return failures **as text the model can read** (an observation), not as a crash.
3. Make the message **actionable**: say what was wrong and what is allowed.

Watch a model pass a string where a number is required, read the error, and repair the call.

In [ ]:
result = run_tool_loop(ScriptedRepairModel(), "What is 2 + 3?", registry)
pd.DataFrame(result["log"])

In [ ]:
# The validator on its own. These are the messages a model would see.
schema = registry.tools["add"]["schema"]
for args in [{"a": 1, "b": 2}, {"a": "1", "b": 2}, {"a": 1}, {"a": 1, "b": 2, "c": 3}]:
    print(f"{str(args):30} -> {validate_args(schema, args) or 'valid'}")

**Checkpoint 1.**
1. Why return the error to the model instead of raising an exception in the harness?
2. What would a *bad* error message look like (hint: `Error 500`) and why does it hurt recovery?
3. Add an `enum` for an `operation` argument and see what the validator says for an unknown value.

<details>
<summary><b>Show answers to Checkpoint 1</b></summary>

1. **Why return the error to the model instead of raising an exception?** Because the model is the one that can fix it. It chose the arguments, so when it reads
   `argument 'a' must be number, got str ('two')` it can retry with a number. An exception would stop the whole run, and the model would never see what went wrong.
   Reserve exceptions for bugs in the harness itself.
2. **A bad message looks like `Error 500`.** It says nothing about the cause, the fix or the valid options, so the model guesses: it repeats the same call or gives up.
   A good message states what was wrong and what is allowed, for example `Unknown tool 'multiply'. Available tools: ['add', 'word_count']`.
3. **Add an `enum`:**
   ```python
   schema = {"type": "object", "properties": {"operation": {"type": "string", "enum": ["add", "sub"]}}, "required": ["operation"]}
   validate_args(schema, {"operation": "mul"})   # ["argument 'operation' must be one of ['add', 'sub']"]
   validate_args(schema, {"operation": "add"})   # []  (valid)
   ```
   The error lists the allowed values, which is exactly what the model needs to repair the call.
</details>

## 3. Tool design: you are writing a prompt for a non-deterministic caller

**Concept.** A normal API is called by deterministic code that follows the docs exactly. A tool is called by a **model** that reads your description
and *guesses*. Anthropic's engineering guide on tools for agents makes the same point: tools must be designed for how models behave.

| Principle | Weak | Strong |
|---|---|---|
| **Name by intent** | `proc_req` | `search_notes` |
| **Describe when to use it** | "Searches stuff" | "Search the user's notes by keyword. Use when the user asks about their own notes, meetings or plans." |
| **Return what the model needs** | raw 40 KB JSON blob | the 5 relevant fields, human-readable names instead of opaque ids |
| **Helpful errors** | `Error 500` | "No note with id 9. Valid ids: 1, 2, 3. Try search_notes first." |
| **Bound the output** | return everything | pagination, `limit`, truncation with a hint ("showing 10 of 240, refine your query") |
| **Fewer, sharper tools** | `get_user`, `fetch_user`, `lookup_user` | one `find_user` |
| **Mark risk** | all tools look equal | separate read-only from side-effecting; irreversible ones need approval |
| **Idempotency** | retry sends two emails | safe to retry, or return a request id |

**Parallel calls.** When two calls are independent, a model can request both in **one turn**, saving a round trip. The harness runs them and returns both results.

In [ ]:
result = run_tool_loop(ScriptedParallelModel(), "Add 10 and 5, and count the words in 'tool use is a loop'.", registry)
print("assistant turns:", sum(1 for m in result["messages"] if m["role"] == "assistant"), "| tool results:", sum(1 for m in result["messages"] if m["role"] == "tool"))

**Checkpoint 2.** Take the description of `send_email` in `mini_mcp_server.py`. What does it tell the model about *when not* to call it? Rewrite one weak description from the table above for a tool of your own.

<details>
<summary><b>Show answers to Checkpoint 2</b></summary>

**What does `send_email`'s description say about when not to call it?** It says the action is **"Irreversible"** and to use it **"only when the user explicitly asks"**.
So the model is told two things: the cost of a mistake is high, and the trigger must be an explicit user request, not its own initiative or text found in a document.
(Our security demo shows that this wording alone is not enough protection. It must be backed by an approval gate.)

**A rewrite of a weak description.**
- Weak: `"Searches stuff"`
- Strong: `"Search the user's notes by keyword. Use when the user asks about their own notes, meetings or plans. Returns up to 5 matches as 'id: text' lines, or 'no matching notes'. Do not use for web search."`

The strong version says **what** it does, **when** to use it, **what comes back** and **when not** to use it.
</details>

## 4. MCP: the architecture

**Concept.** MCP is an open protocol (introduced by Anthropic in late 2024) that standardizes how AI applications connect to tools and data.

| Role | What it is | Example |
|---|---|---|
| **Host** | the AI application the user runs | Claude Desktop, an IDE, your own agent |
| **Client** | a connector inside the host, one per server | the code in `MiniMCPClient` |
| **Server** | a program exposing capabilities | a notes server, a GitHub server |

**Three primitives** (who controls each is the key idea):

| Primitive | Controlled by | Purpose | Analogy |
|---|---|---|---|
| **Tools** | the **model** decides to call | actions and computations, may have side effects | a POST / function call |
| **Resources** | the **application** decides to attach | read-only context identified by URI (`notes://2`) | a GET / file |
| **Prompts** | the **user** picks | reusable templates | a slash command |

**Two layers.** The **data layer** is JSON-RPC 2.0 messages (`initialize`, `tools/list`, `tools/call`, ...). The **transport layer** moves them:
**stdio** (client launches the server as a local subprocess) or **HTTP** (remote servers, with authorization).

**Lifecycle:** `initialize` (negotiate protocol version and capabilities) then `notifications/initialized`, then normal use, then shutdown.

> **Version note.** This lab speaks the **2025-06-18** revision of the protocol: an `initialize` handshake, then `notifications/initialized`.
> The official architecture overview now documents a newer revision (**2026-07-28**) that is **stateless**: every request carries the protocol version and
> client capabilities in its `_meta` field, discovery uses a `server/discover` request, change notifications are opt-in through `subscriptions/listen`,
> and the client features **sampling** and **logging** are deprecated (elicitation remains). The primitives and the `tools/list` and `tools/call`
> shapes you learn here carry over. Always check the spec revision your SDK targets.

## 5. Lab: speak MCP by hand

We launch `mini_mcp_server.py` as a subprocess and talk JSON-RPC to it. No SDK. Watch the **wire log**: that is the whole protocol.

In [ ]:
client = MiniMCPClient("mini_mcp_server.py")
info = client.initialize()
print("server:", info["serverInfo"], "| protocol:", info["protocolVersion"])
print("capabilities:", info["capabilities"])

In [ ]:
tools = client.list_tools()
pd.DataFrame([{"name": t["name"], "description": t["description"][:60]} for t in tools])

In [ ]:
print(client.call_tool("add", {"a": 2, "b": 3}))
print(client.call_tool("search_notes", {"query": "planning"}))
print(client.call_tool("word_count", {"text": 123}))        # wrong type: the server reports a tool error, not a crash

In [ ]:
# The raw wire log for the first few messages. Arrows show direction.
for direction, msg in client.wire[:6]:
    print(direction, json.dumps(msg)[:150])

**Two kinds of error, on purpose.**
- A **protocol error** (unknown method or tool): a JSON-RPC `error` object. It is for the *client software*.
- A **tool execution error** (bad arguments, API failure): a normal result with `isError: true`. It is for the *model*, so it can recover.

Try it: call a tool name that does not exist and read the exception, then compare with the `word_count` case above.

In [ ]:
try:
    client.call_tool("does_not_exist", {})
except RuntimeError as e:
    print("protocol-level error ->", e)

# Resources: read-only context addressed by URI.
print(client.request("resources/list")["resources"][:2])
print(client.request("resources/read", {"uri": "notes://2"})["contents"][0]["text"])

### Reading the whole wire log, line by line

Every message that crossed the pipe was saved in `client.wire`. The cell below prints **all** of it (not just the first six) and adds a plain-English label to each line.
Run it, then use the guide underneath to read it.

In [ ]:
def describe(msg):
    # Classify one JSON-RPC message in plain English.
    if "method" in msg and "id" not in msg:
        return "NOTIFICATION (no id, so no reply)"
    if "method" in msg:
        return f"REQUEST #{msg['id']}: {msg['method']}"
    if "error" in msg:
        return f"PROTOCOL ERROR for #{msg['id']} (code {msg['error']['code']}), meant for the client software"
    if msg["result"].get("isError"):
        return f"RESULT for #{msg['id']}: the tool FAILED (isError true), the model can read it"
    return f"RESULT for #{msg['id']}"

for i, (direction, msg) in enumerate(client.wire):
    side = "client -> server" if direction == "->" else "server -> client"
    print(f"{i:2d}  {side}  {describe(msg)}")
    print(f"      {json.dumps(msg)[:140]}")

**Guide to the log**

| Lines | What you see | What it means |
|---|---|---|
| first two | `initialize` request, then a reply | The **handshake**. The client states its protocol version and capabilities, and the server answers with its own plus its name. |
| next | `notifications/initialized`, no `id` | A **notification**: the client announces "I am ready" and does not wait for a reply. |
| next two | `tools/list` and its reply | **Discovery.** The reply lists each tool's name, description and schema, the same three parts from section 1. |
| each `tools/call` | request with `name` and `arguments`, then a reply with `content` | A **tool call**. The reply holds a list of `content` blocks and `isError`. |
| `isError: true` | a normal reply, but the text is an error message | A **tool failure** (here, a number was passed where text was expected). It comes back as a result so the **model** can read it and fix the call. |
| `"error": {...}` | a reply with `error` instead of `result` | A **protocol failure** (here, an unknown tool name, code `-32602`). It is for the **client software**, not the model. |

**Three rules to remember**
1. The reply repeats the request's **`id`**. That is how replies are matched to requests, even if several are in flight.
2. A message with **no `id`** is a notification. Nobody answers it.
3. The server prints exactly **one JSON object per line** to stdout and the client reads one line at a time. That is why a stdio server must never print anything else to stdout.

**Checkpoint 3.**
1. Why does a *notification* (`notifications/initialized`) have no `id` and get no reply?
2. Why must a stdio server never `print()` debug output to stdout?
3. Which primitive would you use to expose a read-only database schema? Which for "delete a record"?

<details>
<summary><b>Show answers to Checkpoint 3</b></summary>

1. **Why does a notification have no `id` and get no reply?** In JSON-RPC a notification is fire-and-forget: the sender announces something (here, "I am ready") and does not wait.
   There is nothing to return, and with no `id` a reply could not be matched to any request anyway.
2. **Why must a stdio server never `print()` debug output to stdout?** Stdout **is** the protocol channel: the client reads one JSON message per line from it.
   A stray line is not valid JSON, so the client's `json.loads` fails or the exchange desynchronizes. Write logs to **stderr** instead.
3. **Which primitive for what?** A read-only database schema is **context**, so expose it as a **resource** (for example `db://schema`), which the application chooses to attach.
   "Delete a record" is an **action with side effects**, so it is a **tool**, which the model decides to call and which should sit behind an approval gate.
</details>

## 6. Plug MCP tools into the agent loop

The bridge is small: list the server's tools, wrap each in a registry entry whose function calls `client.call_tool`.
From the model's point of view an MCP tool is indistinguishable from a local one.

In [ ]:
mcp_registry = ToolRegistry()
for t in client.list_tools():
    mcp_registry.register(
        t["name"], t["description"], t["inputSchema"],
        fn=lambda _n=t["name"], **kw: client.call_tool(_n, kw)[0],       # forward the call over MCP
        untrusted_output=(t["name"] == "search_notes"),                   # notes may contain attacker-controlled text
        sensitive=(t["name"] == "send_email"),                            # irreversible side effect
    )

print("tools available to the agent:", sorted(mcp_registry.tools))

## 7. The official SDK

Hand-written clients teach the protocol, but in practice use the SDK. An `MCPServer` (called `FastMCP` in SDK v1) derives name, description and JSON Schema
from the function signature and docstring (open `sdk_server.py` and compare it with `mini_mcp_server.py`). The SDK client below launches that server over stdio.

In [ ]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

params = StdioServerParameters(command=sys.executable, args=["sdk_server.py"])
async with stdio_client(params) as (read, write):
    async with ClientSession(read, write) as session:
        await session.initialize()
        listed = await session.list_tools()
        print("tools:", [t.name for t in listed.tools])
        print("schema for add:", json.dumps(listed.tools[0].input_schema))   # camelCase inputSchema on the wire, snake_case in the Python SDK
        out = await session.call_tool("add", {"a": 2, "b": 3})
        print("add(2, 3) ->", out.content[0].text)

Notice the schema was generated from `def add(a: float, b: float) -> float` plus its docstring. Less boilerplate, fewer mismatches between code and schema.

## 8. Security: tool results are untrusted input

**The core problem: prompt injection.** A model reads instructions and data in the same channel. If a tool returns text an attacker controls
(a web page, an email, a shared doc, a ticket), that text can *contain instructions*, and a gullible model may obey them.
This is the most important security fact about tool-using agents.

**Demo.** Note 3 in our server is poisoned. A user simply asks the agent to summarize their meeting notes. Watch a naive harness leak data.

In [ ]:
def run_attack(approve):
    c = MiniMCPClient("mini_mcp_server.py"); c.initialize()
    reg = ToolRegistry()
    for t in c.list_tools():
        reg.register(t["name"], t["description"], t["inputSchema"], fn=lambda _n=t["name"], **kw: c.call_tool(_n, kw)[0],
                     untrusted_output=(t["name"] == "search_notes"), sensitive=(t["name"] == "send_email"))
    res = run_tool_loop(GulliblePlannerModel(), "Summarize my notes about the meeting.", reg, approve=approve)
    leaked = c.call_tool("send_email", {"to": "probe@example.com", "body": "probe"})  # just to ensure server is alive
    c.close()
    return res

print("=== NAIVE harness: no approval gate ===")
naive = run_attack(approve=None)

The agent emailed the contents of every note to the attacker, and the user never asked for an email. Now add a **defense**:
sensitive tools require **human approval** whenever untrusted content has entered the context (a simple form of *taint tracking*).

In [ ]:
def human_approval(call, tainted):
    # In a real app this is a confirmation dialog showing the exact action. Here we simulate the human clicking "deny".
    print(f"      >> approval requested for {call['name']} to={call['args'].get('to')} (context tainted={tainted}) -> DENIED by user")
    return False

print("=== DEFENDED harness: approval gate on sensitive tools ===")
safe = run_attack(approve=human_approval)

**Defense in depth** (no single layer is enough):

| Layer | What it does |
|---|---|
| **Least privilege** | give the agent only the tools the task needs; scope tokens narrowly; prefer read-only |
| **Approval gates** | humans confirm irreversible or data-leaving actions, showing the exact action |
| **Taint tracking** | once untrusted content is in context, restrict dangerous tools |
| **Isolation** | separate the model that reads untrusted content from the one that can act ("dual LLM" patterns) |
| **Output handling** | delimit and label tool output as data; never let it modify system instructions |
| **Server vetting** | treat third-party MCP servers like third-party code: review, pin versions, restrict network and filesystem |
| **Tool poisoning awareness** | a malicious server can hide instructions in a tool *description*; descriptions are untrusted too |
| **Audit logs** | record every call so incidents can be investigated |

**Checkpoint 4.** The gate here keys on `sensitive` plus `tainted`. What attack still works if the attacker's text only asks the agent to call a tool that is *not* marked sensitive (for example one that writes to a public place)?

<details>
<summary><b>Show answers to Checkpoint 4</b></summary>

**The gap:** the gate only protects tools **marked** `sensitive`. If the attacker's text asks the agent to use a tool that is not marked, but still moves data out,
the attack goes straight through. Examples:
- a tool that **writes to a public place** (a public wiki page, a shared document, a public gist);
- a **fetch-URL** tool, where the data is hidden in the query string of a request to the attacker's server;
- a tool that is simply **mislabeled** by whoever registered it.

**Stronger defenses:**
- Classify tools by **capability** (can it send data outside, write somewhere others can read, or change state?), not by a hand-set flag.
- Once the context is **tainted** by untrusted content, require approval for **any** tool call with an outward effect, or restrict the agent to read-only tools.
- Keep an **egress allow-list** of destinations the agent may reach.
- Show the human the **exact action** (destination and payload) when asking for approval.
- Apply least privilege so the agent has few outward-facing tools to abuse in the first place.
</details>

In [ ]:
client.close()   # shut down the first server process

## 9. Exercises

1. **Add a tool** to `mini_mcp_server.py` (for example `list_notes` with a `limit` argument). Re-run the lab and watch it appear in `tools/list`.
2. **Pagination:** make `search_notes` return at most 2 results plus a hint such as "showing 2 of 5, refine your query". Why does the hint matter?
3. **Argument repair:** extend `validate_args` to support `minimum` and `maxLength`. Write a scripted model that violates each.
4. **Prompts primitive:** implement `prompts/list` and `prompts/get` in the mini server.
5. **Timeouts and retries:** make a slow tool and add a timeout in `ToolRegistry.call`. Which errors are safe to retry?
6. **Stronger defense:** block *all* tool calls that include text copied verbatim from an untrusted result. What false positives would it cause?
7. **Compare servers:** run the same client calls against `sdk_server.py` using the SDK client and diff the `tools/list` output with the mini server.
8. **Evaluate your tools:** write 10 tasks and measure how often a model picks the right tool with your descriptions (see the Evals course). Rewrite descriptions and re-measure.

## 10. Glossary

| Term | Meaning |
|---|---|
| **Tool / function calling** | the model emits a structured request; the harness executes it |
| **JSON Schema** | the format used to describe a tool's inputs |
| **MCP** | Model Context Protocol, an open standard for connecting AI apps to tools and data |
| **Host / client / server** | the AI app / its per-server connector / the capability provider |
| **Tools / resources / prompts** | model-controlled actions / app-controlled context / user-selected templates |
| **JSON-RPC 2.0** | the request/response message format MCP uses |
| **stdio transport** | client launches the server locally and talks over stdin/stdout |
| **HTTP transport** | remote servers over the network, with authorization |
| **Capability negotiation** | client and server agree on features during `initialize` |
| **isError** | flag marking a tool execution error that the model should see |
| **Prompt injection** | untrusted text containing instructions that hijack the model |
| **Tool poisoning** | malicious instructions hidden in a tool description or result |
| **Taint tracking** | marking context as contaminated by untrusted input |
| **Least privilege** | granting only the access a task needs |

## 11. Cheat sheet
```
Loop       : model requests -> harness validates -> tool runs -> result back as a message
Errors     : validate first; return actionable text to the MODEL; protocol errors are for the CLIENT
Design     : name by intent, describe WHEN to use, bounded outputs, helpful errors, few sharp tools
MCP        : host -> client -> server;  tools (model) / resources (app) / prompts (user)
Wire       : JSON-RPC 2.0, one message per line on stdio;  initialize -> notifications/initialized -> tools/list -> tools/call
Security   : tool results are UNTRUSTED. least privilege + approval gates + taint tracking + vetted servers
```

## 12. Reading list
- Anthropic, Writing effective tools for agents: https://www.anthropic.com/engineering/writing-tools-for-agents
- MCP architecture overview (community-hosted mirror of the docs): https://modelcontextprotocol.info/docs/2026-07-28/learn/architecture/
- MCP definitive guide (DEV Community): https://dev.to/kuldeep_paul/the-definitive-guide-to-the-model-context-protocol-4fpn
- Official MCP site and specification: https://modelcontextprotocol.io *(from memory, not re-checked: this is the canonical home of the spec, check it loads)*
- Python SDK: search "modelcontextprotocol python-sdk" on GitHub for the official repository.
- Related courses in this series: Harness Engineering (the loop around tools) and Evals (measuring tool-use quality).

## 13. My experiments log

| # | Change | What happened | What I learned |
|---|--------|---------------|----------------|
| 0 | baseline | | |
| 1 | | | |

## Where next
1. Build a real MCP server around something you use (your notes, a database) and connect it to an MCP-capable app.
2. Add tool-selection evals to measure description quality.
3. Revisit security: threat-model one of your own agents.